# v2.3.5 SUMA 遠端模型啟動器：初次啟動與後續切換

先啟用 Colab GPU，在第一個表單選擇模型、MMProj、上下文並執行。

之後換模型只執行下方第二個表單。兩個程式格在 Colab 均設定為表單檢視，預設隱藏程式碼；下載進度與服務輸出仍會顯示。這只是介面摺疊，不是加密或禁止查看原始碼。

### 支援模型清單與特色

本啟動器內建以下四款經最佳化的開源 GGUF 模型，啟動時提供 **64K**、**128K** 與 **256K** 三種上下文長度選項（請依實際硬體配置選用，以避免 GPU 記憶體不足）：

1. **Gemma 4 12B QAT Q4_0**
   - **模型特色**：Google 最新一代輕量高效能指令微調模型，經量化感知訓練（QAT）處理，兼顧低顯卡記憶體佔用與高精確度；支援多模態視覺投影（MMProj）。
   - **支援多模態**：是（內含 MMProj）
   - **原生架構上下文**：最高 256K tokens（本啟動器支援最高配置至 256K）

2. **Qwen3.5-9B Q5_K_M**
   - **模型特色**：阿里通義千問優秀的輕量多語言模型，採用 Q5_K_M 中高精度量化，推論流暢、中文理解與綜合推理表現均衡；亦支援搭配 MMProj 進行視覺理解。
   - **支援多模態**：是（倉庫提供 `mmproj-BF16.gguf` / `mmproj-F16.gguf`，勾選後自動下載載入）
   - **原生架構上下文**：最高 262,144 tokens（256K，本啟動器支援最高配置至 256K）

3. **Qwen3.8-27B UD-Q2_K_XL**
   - **模型特色**：27B 中大參數量模型，透過極致的 UD-Q2 超低位元量化技術，讓 16GB 顯卡記憶體（如 Colab T4）也能載入大參數量模型進行高難度推理；亦支援多模態視覺投影。
   - **支援多模態**：是（倉庫提供 `mmproj-BF16.gguf` / `mmproj-F16.gguf`，勾選後自動下載載入）
   - **原生架構上下文**：最高 262,144 tokens（256K，本啟動器支援最高配置至 256K）

4. **Inkling-2-Mini UD-Q2_K_XL**
   - **模型特色**：Thinking Machines 的 MoE 多模態模型，採用 Unsloth Dynamic 3.0 UD-Q2_K_XL 量化，兼顧低顯卡記憶體佔用、綜合推理與視覺理解能力。
   - **支援多模態**：是（倉庫提供 `mmproj-BF16.gguf` / `mmproj-F16.gguf`，勾選後自動下載載入）
   - **原生架構上下文**：最高 256K tokens（本啟動器支援最高配置至 256K）

### 注意事項

- 移除兩個 Uncensored 選項；保留四款公開來源模型選項，下載不讀取／傳送 HF_TOKEN。倉庫 API 回報 gated 或 private 時停止，不進入授權設定流程。
- 保留 llama.cpp b12651 官方預編譯套件、所有下載進度與斷點續傳；不編譯、不做內容雜湊。既有 GGUF 只讀 4-byte 檔頭，這不是完整內容驗證。
- 保留 SUMA Remote AI Bridge 0.3.6.0 Beta 自動下載與啟動（啟用 `--auto-approve` 自動接受連線，無需人工驗證）及其實際輸出；輸出可能含配對碼或 Token，請勿公開。
- 模型固定存放於 `/content/models` 路徑，不隨 llama.cpp 版本變動且不刪除舊模型。要沿用檔案須使用原 Notebook 的同一執行階段，不要重設。另一份 Notebook 不保證共用原 VM。
- 上下文按選擇保留，記憶體不足會報錯。若選取的模型倉庫不含 MMProj，將發出提醒並自動改為純文字模式（略過 MMProj），避免流程中斷。
- 可另開程式格執行 `show_suma_log()` 或 `stop_services()`。
- 請分別執行需要的表單；「執行全部」會先啟動第一個模型，再切換至第二個表單的選擇。
- 完成 Notebook 格式、Python 語法、三項下拉選單及切換流程的離線檢查；未在 Colab GPU 完整實測模型載入與 SUMA 配對。


In [ ]:
# @title 選擇模型、MMProj 與上下文，然後執行
MODEL = "Gemma 4 12B QAT Q4_0" # @param ["Gemma 4 12B QAT Q4_0", "Qwen3.5-9B Q5_K_M", "Qwen3.8-27B UD-Q2_K_XL", "Inkling-2-Mini UD-Q2_K_XL"]
LOAD_MMPROJ = True # @param {type:"boolean"}
CONTEXT = "64K" # @param ["64K", "128K", "256K"]

import codecs
import json
import os
import platform
import re
import shlex
import shutil
import signal
import socket
import subprocess
import sys
import tarfile
import threading
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

_VERSION = 'b12651'
_ROOT = Path('/content/llama-direct-b12651') if Path('/content').exists() else Path.home()/'llama-direct-b12651'
_MODELS_ROOT = Path('/content/models') if Path('/content').exists() else Path.home()/'models'
_KNOWN_ASSETS = {
    'b12651': [
        {'name':'llama-b12651-bin-ubuntu-cuda-12.8-x64.tar.gz','size':220195495,'browser_download_url':'https://github.com/ggml-org/llama.cpp/releases/download/b12651/llama-b12651-bin-ubuntu-cuda-12.8-x64.tar.gz'},
        {'name':'cudart-llama-b12651-bin-ubuntu-cuda-12.8-x64.tar.gz','size':594371475,'browser_download_url':'https://github.com/ggml-org/llama.cpp/releases/download/b12651/cudart-llama-b12651-bin-ubuntu-cuda-12.8-x64.tar.gz'},
        {'name':'llama-b12651-bin-ubuntu-cuda-13.5-x64.tar.gz','size':224478514,'browser_download_url':'https://github.com/ggml-org/llama.cpp/releases/download/b12651/llama-b12651-bin-ubuntu-cuda-13.5-x64.tar.gz'},
        {'name':'cudart-llama-b12651-bin-ubuntu-cuda-13.5-x64.tar.gz','size':444078776,'browser_download_url':'https://github.com/ggml-org/llama.cpp/releases/download/b12651/cudart-llama-b12651-bin-ubuntu-cuda-13.5-x64.tar.gz'},
        {'name':'llama-b12651-bin-ubuntu-cuda-13.5-arm64.tar.gz','size':219392133,'browser_download_url':'https://github.com/ggml-org/llama.cpp/releases/download/b12651/llama-b12651-bin-ubuntu-cuda-13.5-arm64.tar.gz'},
        {'name':'cudart-llama-b12651-bin-ubuntu-cuda-13.5-arm64.tar.gz','size':559873124,'browser_download_url':'https://github.com/ggml-org/llama.cpp/releases/download/b12651/cudart-llama-b12651-bin-ubuntu-cuda-13.5-arm64.tar.gz'}
    ]
}
_CTX = {'64K':65536,'128K':131072,'256K':262144}[CONTEXT]
_PORT = 8081
_TIMEOUT = 900
_NETWORK_TIMEOUT = 45
_RETRIES = 3
_MODELS = {
    'Gemma 4 12B QAT Q4_0': {'repo':'google/gemma-4-12B-it-qat-q4_0-gguf','file':'gemma-4-12b-it-qat-q4_0.gguf','alias':'gemma-4-12b','mmproj':'mmproj-gemma-4-12b-it-qat-q4_0.gguf'},
    'Qwen3.5-9B Q5_K_M': {'repo':'unsloth/Qwen3.5-9B-GGUF','file':'Qwen3.5-9B-Q5_K_M.gguf','alias':'qwen3.5-9b'},
    'Qwen3.8-27B UD-Q2_K_XL': {'repo':'unsloth/Qwen3.8-27B-GGUF','file':'Qwen3.8-27B-UD-Q2_K_XL.gguf','alias':'qwen3.8-27b-q2kxl'},
    'Inkling-2-Mini UD-Q2_K_XL': {'repo':'unsloth/Inkling-2-Mini-GGUF','file':'Mainline_Support/Inkling-2-Mini-UD-Q2_K_XL.gguf_file','alias':'inkling-2-mini-q2kxl','mmproj':'mmproj-BF16.gguf'},
}
_SELECTED = _MODELS[MODEL]

def _secret(name):
    if os.environ.get(name):
        return os.environ[name]
    if 'google.colab' in sys.modules:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            pass
    return None

def _human(value):
    if value is None:
        return '未知'
    value = float(value)
    for unit in ('B','KiB','MiB','GiB','TiB'):
        if abs(value)<1024 or unit=='TiB':
            return f'{value:.2f} {unit}'
        value /= 1024

def _clock(seconds):
    if seconds is None:
        return '--:--'
    seconds=max(0,int(seconds))
    return f'{seconds//3600:02d}:{seconds//60%60:02d}:{seconds%60:02d}'

class _Progress:
    def __init__(self,label,total=None,initial=0):
        self.label,self.total,self.done=label,total,initial
        self.initial=initial
        self.phase='連線／等待回應'
        self.started=self.last_data=time.monotonic()
        self.event=threading.Event();self.lock=threading.Lock()
    def __enter__(self):
        print('\n▶ '+self.label,flush=True);self.render()
        self.thread=threading.Thread(target=self.tick,daemon=True);self.thread.start()
        return self
    def configure(self,total=None,initial=0,phase=None):
        if phase is None:
            phase='下載中'
        with self.lock:
            self.total,self.done,self.initial=total,initial,initial
            self.phase=phase;self.started=self.last_data=time.monotonic()
    def update(self,count):
        with self.lock:
            self.done+=count;self.last_data=time.monotonic()
    def render(self,final=None):
        with self.lock:
            now=time.monotonic();elapsed=now-self.started
            speed=max(0,self.done-self.initial)/max(elapsed,0.001)
            percentage=f'{min(100,self.done/self.total*100):6.2f}%' if self.total else '   --%'
            eta=max(0,self.total-self.done)/speed if self.total and speed>0 else None
            phase=self.phase
            if phase=='下載中' and now-self.last_data>=5:
                phase=f'等待資料 {int(now - self.last_data)}s'
            elapsed_label = '已用'
            line=f'{percentage} | {_human(self.done)} / {_human(self.total)} | {_human(speed)}/s | {elapsed_label} {_clock(elapsed)} | ETA {_clock(eta)} | {final or phase}'
            print('\r'+line.ljust(155),end='',flush=True)
    def tick(self):
        while not self.event.wait(1):
            self.render()
    def __exit__(self,kind,value,traceback):
        self.event.set();self.thread.join(timeout=2)
        self.render('完成' if kind is None else '中斷／失敗');print(flush=True)

class _SafeRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self,request,fp,status,message,headers,newurl):
        redirected=super().redirect_request(request,fp,status,message,headers,newurl)
        if redirected and urllib.parse.urlparse(request.full_url).hostname!=urllib.parse.urlparse(newurl).hostname:
            redirected.remove_header('Authorization')
        return redirected

_OPENER=urllib.request.build_opener(_SafeRedirect())

def _request_url(url):
    if urllib.parse.urlsplit(url).hostname!='huggingface.co':return url
    parts=urllib.parse.urlsplit(url)
    query=urllib.parse.parse_qsl(parts.query,keep_blank_values=True)
    query=[(key,value) for key,value in query if key!='_colab_retry']
    query.append(('_colab_retry',str(time.time_ns())))
    return urllib.parse.urlunsplit(parts._replace(query=urllib.parse.urlencode(query)))

def _error(exc):
    if isinstance(exc,urllib.error.HTTPError):
        host=urllib.parse.urlparse(str(exc.url)).hostname or ''
        if 'huggingface.co' in host and exc.code in (401,403):
            return f'HTTP {exc.code}：匿名下載失敗。本版不處理需授權來源，請改選其他模型或檢查網路。'
        if host=='api.github.com' and exc.code in (403,429):
            return 'GitHub API 權限／限流問題，可提供 GITHUB_TOKEN 後重試。'
        return f'HTTP {exc.code}：{exc.reason}'
    return str(exc)

def _read_json(url,label,token=None):
    headers={'User-Agent':'colab-simple-model-runner','Accept':'application/json','Cache-Control':'no-cache'}
    if token:headers['Authorization']='Bearer '+token
    for attempt in range(_RETRIES):
        try:
            with _Progress(label) as progress:
                with _OPENER.open(urllib.request.Request(_request_url(url),headers=headers),timeout=_NETWORK_TIMEOUT) as response:
                    length=response.headers.get('Content-Length');progress.configure(int(length) if length else None)
                    data=bytearray()
                    while True:
                        block=response.read1(64*1024)
                        if not block:break
                        data.extend(block);progress.update(len(block))
            return json.loads(data)
        except (urllib.error.URLError,TimeoutError,OSError) as exc:
            if (isinstance(exc,urllib.error.HTTPError) and exc.code in (400,401,403,404)) or attempt==_RETRIES-1:
                raise RuntimeError(_error(exc)) from exc
            print(f'{_error(exc)}；重試 {attempt + 2}/{_RETRIES}',flush=True);time.sleep(2**attempt)

def _size_matches(path,size=None):
    path=Path(path)
    return path.is_file() and path.stat().st_size>0 and (size is None or path.stat().st_size==size)

def _download(url,dest,label=None,size=None,token=None):
    dest=Path(dest);dest.parent.mkdir(parents=True,exist_ok=True)
    if _size_matches(dest,size):
        print(f'✓ 沿用 {dest.name}（{_human(dest.stat().st_size)}；不做雜湊）',flush=True);return dest
    partial=dest.with_name(dest.name+'.part');meta=dest.with_name(dest.name+'.part.json')
    try:saved=json.loads(meta.read_text()) if meta.exists() else {}
    except (ValueError,OSError):saved={}
    if saved.get('url')!=url or (size is not None and partial.exists() and partial.stat().st_size>size):
        partial.unlink(missing_ok=True);saved={}
    for attempt in range(_RETRIES):
        offset=partial.stat().st_size if partial.exists() else 0
        if offset and size is not None and offset==size:
            partial.replace(dest);meta.unlink(missing_ok=True)
            print('✓ 下載完成：',dest.name,flush=True);return dest
        headers={'User-Agent':'colab-simple-model-runner','Accept-Encoding':'identity','Cache-Control':'no-cache'}
        if token:headers['Authorization']='Bearer '+token
        if offset:
            headers['Range']=f'bytes={offset}-'
            if saved.get('validator'):headers['If-Range']=saved['validator']
        print('下載來源：',url,flush=True)
        try:
            with _Progress(label or dest.name,size,offset) as progress:
                with _OPENER.open(urllib.request.Request(_request_url(url),headers=headers),timeout=_NETWORK_TIMEOUT) as response:
                    if getattr(response,'status',200)==206:
                        match=re.fullmatch(r'bytes (\d+)-(\d+)/(\d+|\*)',response.headers.get('Content-Range',''))
                        if not match or int(match.group(1))!=offset:
                            partial.unlink(missing_ok=True);raise RuntimeError('續傳範圍錯誤，移除暫存後重試。')
                        total=int(match.group(3)) if match.group(3)!='*' else size
                    else:
                        if offset:print('\n未接受續傳或來源已更新，從頭下載。',flush=True)
                        offset=0;length=response.headers.get('Content-Length');total=int(length) if length else size
                    if size is not None and total is not None and size!=total:
                        partial.unlink(missing_ok=True);raise RuntimeError('回應大小與預期不符。')
                    validator=response.headers.get('ETag') or response.headers.get('Last-Modified')
                    if validator and validator.startswith('W/'):validator=response.headers.get('Last-Modified')
                    saved={'url':url,'validator':validator};meta.write_text(json.dumps(saved))
                    progress.configure(total,offset)
                    with partial.open('ab' if offset else 'wb') as output:
                        while True:
                            block=response.read1(1024*1024)
                            if not block:break
                            output.write(block);progress.update(len(block))
                    if total is not None and partial.stat().st_size!=total:
                        raise RuntimeError(f'下載未完成：{partial.stat().st_size}/{total} bytes')
            if not _size_matches(partial,size):raise RuntimeError('下載檔案為空或大小不符。')
            partial.replace(dest);meta.unlink(missing_ok=True)
            print(f'✓ 下載完成：{dest.name}（{_human(dest.stat().st_size)}）',flush=True);return dest
        except (urllib.error.URLError,TimeoutError,OSError,RuntimeError) as exc:
            if isinstance(exc,urllib.error.HTTPError):
                if exc.code in (400,401,403,404):raise RuntimeError(_error(exc)) from exc
                if exc.code==416:partial.unlink(missing_ok=True);meta.unlink(missing_ok=True);saved={}
            if attempt==_RETRIES-1:raise RuntimeError(_error(exc)) from exc
            print(f'{_error(exc)}；重試 {attempt + 2}/{_RETRIES}',flush=True);time.sleep(2**attempt)

def _extract(archive,dest):
    dest=Path(dest);marker=dest/'.extracted'
    if marker.exists():print('✓ 已解壓縮：',dest.name,flush=True);return
    if not hasattr(tarfile,'data_filter'):raise RuntimeError('請更新 Colab 執行階段以支援安全解壓。')
    dest.mkdir(parents=True,exist_ok=True)
    with tarfile.open(archive,'r:gz') as tar:
        with _Progress('讀取壓縮檔索引：'+Path(archive).name):members=tar.getmembers()
        total=sum(item.size for item in members if item.isfile())
        with _Progress('解壓縮：'+Path(archive).name,total) as progress:
            progress.configure(total,phase='解壓縮中')
            for item in members:
                tar.extract(item,dest,filter='data')
                if item.isfile():progress.update(item.size)
    marker.touch();print('✓ 解壓縮完成：',dest,flush=True)

def _capture(cmd,env=None):
    result=subprocess.run(list(map(str,cmd)),capture_output=True,text=True,errors='replace',env=env,timeout=60)
    if result.returncode:raise RuntimeError(f'退出碼 {result.returncode}\n{result.stdout}\n{result.stderr}')
    return result.stdout+result.stderr

def _env(paths):
    env={key:value for key,value in os.environ.items() if not key.startswith('LLAMA_ARG_') and key not in ('LLAMA_API_KEY','LLAMA24_API_KEY')}
    env['LD_LIBRARY_PATH']=':'.join(dict.fromkeys([*map(str,paths),*filter(None,env.get('LD_LIBRARY_PATH','').split(':'))]))
    key=os.environ.get('LLAMA24_API_KEY','')
    if key:env['LLAMA_API_KEY']=key
    return env

def _candidates(assets,arch,driver):
    candidates=[]
    fallback=[]
    for asset in assets:
        match=re.fullmatch(rf'llama-{re.escape(_VERSION)}-bin-ubuntu-cuda-(\d+)\.(\d+)-{re.escape(arch)}\.tar\.gz',asset['name'])
        if not match:continue
        cuda=tuple(map(int,match.groups()))
        if cuda<=driver:
            candidates.append((cuda,asset))
        elif cuda[0]==driver[0]:
            fallback.append((cuda,asset))
        else:
            print(f"略過 {asset['name']}：高於驅動顯示的 {driver[0]}.{driver[1]}",flush=True)
    if not candidates and fallback:
        fallback.sort(key=lambda pair:pair[0])
        print(f"未找到 <= {driver[0]}.{driver[1]} 套件，嘗試同主版本套件：{fallback[0][1]['name']}",flush=True)
        return [fallback[0][1]]
    return [asset for _,asset in sorted(candidates,key=lambda pair:pair[0],reverse=True)]

def _generate_candidate_assets(version,arch):
    base=f'https://github.com/ggml-org/llama.cpp/releases/download/{version}'
    items=[]
    for cver in ('13.5','12.8','12.4','12.2','12.0','11.8'):
        items.append({'name':f'llama-{version}-bin-ubuntu-cuda-{cver}-{arch}.tar.gz','browser_download_url':f'{base}/llama-{version}-bin-ubuntu-cuda-{cver}-{arch}.tar.gz','size':None})
        items.append({'name':f'cudart-llama-{version}-bin-ubuntu-cuda-{cver}-{arch}.tar.gz','browser_download_url':f'{base}/cudart-llama-{version}-bin-ubuntu-cuda-{cver}-{arch}.tar.gz','size':None})
    return items

def _test_binary(info):
    exe=Path(info['server']);env=_env(info['library_paths'])
    if not exe.is_file():raise RuntimeError('執行檔遺失。')
    print(_capture([exe,'--version'],env))
    if shutil.which('ldd'):
        targets=[exe,*exe.parent.glob('libggml-cuda.so*')]
        for target in targets:
            deps=_capture(['ldd',target],env)
            if 'not found' in deps:raise RuntimeError(deps)
    devices=_capture([exe,'--list-devices'],env);print(devices)
    if not re.search(r'\bCUDA\d+\s*:',devices):raise RuntimeError('沒有可用 CUDA 裝置。')
    return info

def _ensure_llama():
    if sys.platform!='linux' or not shutil.which('nvidia-smi'):
        raise RuntimeError('請先將 Colab 切換為 GPU 執行階段。')
    smi=_capture(['nvidia-smi']);print(smi)
    match=re.search(r'CUDA Version:\s*(\d+)\.(\d+)',smi)
    if not match:raise RuntimeError('無法辨識驅動 CUDA 版本。')
    driver=tuple(map(int,match.groups()));record=_ROOT/'installation.json'
    if record.exists():
        try:
            info=json.loads(record.read_text());version=re.search(r'-cuda-(\d+)\.(\d+)-',info.get('asset',''))
            if info.get('tag')==_VERSION and version and tuple(map(int,version.groups()))<=driver:
                print('▶ 自動沿用已下載的 llama.cpp。',flush=True)
                return _test_binary(info)
        except (RuntimeError,OSError,ValueError,KeyError,subprocess.TimeoutExpired) as exc:
            print('既有安裝未通過檢查，重新選擇預編譯套件：',exc,flush=True)
    arch={'x86_64':'x64','aarch64':'arm64'}.get(platform.machine())
    if not arch:raise RuntimeError('不支援此 CPU 架構。')
    assets=_KNOWN_ASSETS.get(_VERSION)
    if not assets:
        try:
            release=_read_json(f'https://api.github.com/repos/ggml-org/llama.cpp/releases/tags/{_VERSION}','取得官方套件清單',_secret('GITHUB_TOKEN'))
            if release.get('tag_name')==_VERSION and not release.get('draft'):
                assets=release.get('assets',[])
        except Exception as exc:
            print(f'取得 GitHub API 清單失敗（{exc}），改用官方發行直連備援清單...',flush=True)
            assets=_generate_candidate_assets(_VERSION,arch)
    if not assets:
        assets=_generate_candidate_assets(_VERSION,arch)
    by_name={asset['name']:asset for asset in assets}
    candidates=_candidates(assets,arch,driver)
    if not candidates:raise RuntimeError('沒有匹配的官方 CUDA 預編譯套件，不會改為編譯。\n'+'\n'.join(by_name))
    def fetch(asset):
        url=asset['browser_download_url']
        if Path(asset['name']).name!=asset['name'] or not url.startswith(f'https://github.com/ggml-org/llama.cpp/releases/download/{_VERSION}/'):
            raise ValueError('不是指定版本的官方資產。')
        return _download(url,_ROOT/'downloads'/asset['name'],asset['name'],asset.get('size'))
    errors=[]
    for asset in candidates:
        try:
            package=_ROOT/'packages'/asset['name'].removesuffix('.tar.gz');_extract(fetch(asset),package)
            runtime='cudart-'+asset['name']
            if runtime in by_name:_extract(fetch(by_name[runtime]),package/'cuda-runtime')
            servers=list(package.rglob('llama-server'))
            if len(servers)!=1:raise RuntimeError(f'預期唯一 llama-server，實際 {len(servers)} 個。')
            exe=servers[0].resolve();exe.chmod(exe.stat().st_mode|0o111)
            libraries=sorted({str(path.parent.resolve()) for path in package.rglob('*.so*') if path.is_file()})
            info={'tag':_VERSION,'asset':asset['name'],'server':str(exe),'library_paths':list(dict.fromkeys([str(exe.parent),*libraries])),'method':'official-prebuilt-download'}
            _test_binary(info);record.write_text(json.dumps(info,ensure_ascii=False,indent=2));return info
        except (RuntimeError,OSError,tarfile.TarError,subprocess.TimeoutExpired) as exc:
            errors.append(f'{asset["name"]}: {exc}');print('套件無法使用：',exc,flush=True)
    raise RuntimeError('\n\n'.join(errors))

def _gguf(path,main=False):
    path=Path(path)
    if not path.is_file():raise FileNotFoundError(path)
    with path.open('rb') as source:
        if source.read(4)!=b'GGUF':raise RuntimeError('不是 GGUF 檔案：'+str(path))
    if main:
        match=re.search(r'-(\d{5})-of-(\d{5})\.gguf$',path.name)
        if match:
            index,count=map(int,match.groups())
            if index!=1:raise RuntimeError('分片模型必須指定第一片。')
            for number in range(1,count+1):
                name=re.sub(r'-\d{5}-of-\d{5}\.gguf$',f'-{number:05d}-of-{count:05d}.gguf',path.name)
                if not path.with_name(name).is_file():raise RuntimeError('缺少分片：'+name)
    print(f'✓ 檔案就緒：{path.name}（{_human(path.stat().st_size)}；不做雜湊）',flush=True);return path

def _projector(names, required=True):
    candidates=[name for name in names if name.lower().endswith('.gguf') and ('mmproj' in Path(name).name.lower() or 'vision' in Path(name).name.lower()) and not any(word in Path(name).name.lower() for word in ('assistant','mtp','imatrix'))]
    if not candidates:
        if required:raise RuntimeError('此倉庫沒有可辨識的 MMProj；請取消「載入 MMProj」後重新執行。')
        return None
    candidates.sort(key=lambda name:('mmproj' not in name.lower(),not any(word in name.lower() for word in ('f16','fp16')),name))
    print('自動選擇 MMProj：',candidates[0],flush=True);return candidates[0]

def _prepare_files():
    repo=_SELECTED['repo'];folder=_MODELS_ROOT/repo.replace('/','--')
    result={};pending=[];model_name=_SELECTED['file']
    local_model=model_name[:-5] if model_name.endswith('.gguf_file') else model_name
    if (folder/local_model).is_file():result['model']=_gguf(folder/local_model,True)
    else:pending.append(('model',model_name))
    metadata=_ROOT/'metadata'/(repo.replace('/','--')+'.json')
    if LOAD_MMPROJ:
        projector_name=_SELECTED.get('mmproj')
        if not projector_name and folder.exists():
            names=[str(path.relative_to(folder)) for path in folder.rglob('*.gguf')]
            projector_name=_projector(names, required=False)
        if not projector_name and metadata.exists():
            try:
                cached=json.loads(metadata.read_text());projector_name=_projector([item['rfilename'] for item in cached.get('siblings',[])], required=False)
            except (ValueError,KeyError,OSError):pass
        if projector_name:
            if (folder/projector_name).is_file():result['mmproj']=_gguf(folder/projector_name)
            else:pending.append(('mmproj',projector_name))
        else:
            pending.append(('mmproj',None))
    if not pending:
        print('✓ 自動使用本機檔案，不查模型 API、不重掃描內容。',flush=True);return result
    token=None;encoded=urllib.parse.quote(repo,safe='/')
    info=_read_json(f'https://huggingface.co/api/models/{encoded}?blobs=true','取得所選模型檔案清單',token)
    if info.get('private') or info.get('gated') not in (None,False):
        raise RuntimeError('來源目前需要授權，本版僅使用公開模型，請改選其他模型。')
    metadata.parent.mkdir(parents=True,exist_ok=True)
    metadata.write_text(json.dumps(info,ensure_ascii=False));revision=info.get('sha')
    if not revision:raise RuntimeError('模型 API 未提供 commit，無法固定續傳來源。')
    files={item['rfilename']:item for item in info.get('siblings',[])}
    for kind,name in pending:
        if kind=='mmproj' and not name:
            name=_projector(list(files), required=False)
            if not name:
                print('ℹ 此模型倉庫無 MMProj 檔案，自動轉為純文字模式（略過 MMProj）。',flush=True)
                continue
        if name not in files:raise RuntimeError(f'倉庫缺少 {name}。\n可用 GGUF：\n'+'\n'.join(key for key in files if key.endswith('.gguf')))
        local_name=name[:-5] if name.endswith('.gguf_file') else name
        path=folder/local_name;path.parent.mkdir(parents=True,exist_ok=True)
        item=files[name];size=item.get('size') or (item.get('lfs') or {}).get('size')
        url=f'https://huggingface.co/{encoded}/resolve/{revision}/{urllib.parse.quote(name,safe="/")}?download=true'
        if size and not _size_matches(path,size):
            partial=path.with_name(path.name+'.part');meta=path.with_name(path.name+'.part.json')
            try:saved=json.loads(meta.read_text()) if meta.exists() else {}
            except (ValueError,OSError):saved={}
            reusable=partial.stat().st_size if partial.exists() and saved.get('url')==url else 0
            if shutil.disk_usage(path.parent).free<max(0,size-reusable)+1024**3:raise RuntimeError('磁碟空間不足：'+name)
        _download(url,path,('主模型：' if kind=='model' else 'MMProj：')+name,size,token)
        result[kind]=_gguf(path,kind=='model')
    return result

def _identity(pid):
    try:
        proc=Path('/proc')/str(pid);fields=(proc/'stat').read_text().rsplit(')',1)[1].split()
        return {'pid':pid,'start_ticks':fields[19],'exe':str((proc/'exe').resolve(strict=True))}
    except (FileNotFoundError,ProcessLookupError,PermissionError):return None

def _stop_owned(name='server'):
    path=_ROOT/(name+'-process.json')
    if not path.exists():return
    try:saved=json.loads(path.read_text())
    except Exception:saved={}
    pid=saved.get('pid')
    if pid and _identity(pid)==saved:
        try:
            os.kill(pid,signal.SIGTERM)
            for _ in range(50):
                if _identity(pid)!=saved:break
                time.sleep(0.2)
            if _identity(pid)==saved:os.kill(pid,signal.SIGKILL)
        except ProcessLookupError:pass
    try:
        if pid:os.waitpid(pid,os.WNOHANG)
    except (ChildProcessError,OSError):pass
    path.unlink(missing_ok=True)

def _tail(path,limit=14000):
    path=Path(path)
    if not path.exists():return '(尚無日誌)'
    with path.open('rb') as source:
        source.seek(0,2);source.seek(max(0,source.tell()-limit));return source.read().decode('utf-8',errors='replace')

def _health(path='/health'):
    key=os.environ.get('LLAMA24_API_KEY','');headers={'Authorization':'Bearer '+key} if key else {}
    request=urllib.request.Request(f'http://127.0.0.1:{_PORT}{path}',headers=headers)
    opener=urllib.request.build_opener(urllib.request.ProxyHandler({}))
    with opener.open(request,timeout=5) as response:return json.load(response)

def _local_model_path(item):
    name = item['file']
    local_name = name[:-5] if name.endswith('.gguf_file') else name
    return _MODELS_ROOT/item['repo'].replace('/', '--')/local_name

def _sync_models_dir():
    models_dir = _ROOT/'active_models'
    models_dir.mkdir(parents=True, exist_ok=True)
    for name, item in _MODELS.items():
        m_path = _local_model_path(item)
        link_path = models_dir/f"{item['alias']}.gguf"
        if m_path.is_file():
            if not link_path.exists():
                try:
                    link_path.symlink_to(m_path)
                except (OSError, NotImplementedError):
                    try:
                        os.link(m_path, link_path)
                    except OSError:
                        shutil.copyfile(m_path, link_path)
        else:
            if link_path.is_symlink() or link_path.is_file():
                link_path.unlink(missing_ok=True)
    return models_dir

def _model_section(item,paths,models_dir=None):
    # Build one router preset section for a downloaded model; None when not downloaded.
    folder=_MODELS_ROOT/item['repo'].replace('/','--')
    selected=item['alias']==_SELECTED['alias']
    local_model=paths['model'] if selected else _local_model_path(item)
    if not Path(local_model).is_file():return None
    unified=(models_dir or (_ROOT/'active_models'))/f"{item['alias']}.gguf"
    model=unified if unified.is_file() else local_model
    projector=None
    if LOAD_MMPROJ:
        if selected:projector=paths.get('mmproj')
        else:
            name=item.get('mmproj') or (_projector([str(p.relative_to(folder)) for p in folder.rglob('*.gguf')],required=False) if folder.exists() else None)
            if name and (folder/name).is_file():projector=folder/name
    micro=256 if projector else (64 if _CTX>=131072 else 128)
    lines=[f"[{item['alias']}]",f'model = {model}',f'ubatch-size = {micro}']
    if projector:lines.append(f'mmproj = {projector}')
    return lines+['']

def _write_preset(paths):
    # Router mode lists every downloaded model in the unified active_models dir; only one stays loaded (T4 VRAM).
    models_dir=_sync_models_dir()
    threads=max(1,min(8,os.cpu_count() or 2))
    lines=['[*]',f'ctx-size = {_CTX}','parallel = 1',f'threads = {threads}','batch-size = 512',
           'flash-attn = on','cache-type-k = q8_0','cache-type-v = q8_0','fit = on','fit-target = 1024','cache-ram = 0','']
    for item in _MODELS.values():
        section=_model_section(item,paths,models_dir)
        if section:lines+=section
    preset=_ROOT/'models-preset.ini';preset.write_text('\n'.join(lines),encoding='utf-8')
    return preset

def _command(exe,paths,help_text=''):
    if '--models-preset' in help_text:
        return [str(exe),'--models-preset',str(_write_preset(paths)),'--models-max','1','--host','127.0.0.1','--port',str(_PORT)]
    # Fallback for builds without router mode: single model only.
    has_mmproj = LOAD_MMPROJ and 'mmproj' in paths
    micro=256 if has_mmproj else (64 if _CTX>=131072 else 128)
    threads=max(1,min(8,os.cpu_count() or 2))
    cmd=[str(exe),'--model',str(paths['model']),'--alias',_SELECTED['alias'],'--host','127.0.0.1','--port',str(_PORT),
         '--ctx-size',str(_CTX),'--parallel','1','--threads',str(threads),'--batch-size','512','--ubatch-size',str(micro),
         '--flash-attn','on','--cache-type-k','q8_0','--cache-type-v','q8_0','--fit','on','--fit-target','1024','--cache-ram','0']
    cmd+=['--mmproj',str(paths['mmproj'])] if has_mmproj else ['--no-mmproj']
    return cmd

def _router_ready():
    # Ask the router to load the selected model and report whether it is ready.
    alias=_SELECTED['alias']
    status={m.get('id'):(m.get('status') or {}).get('value') for m in _health('/v1/models').get('data',[])}
    current=status.get(alias)
    if current is None:raise RuntimeError(f'路由伺服器未列出模型：{alias}')
    if current=='loaded':return True
    if current not in ('loading',):
        if current not in ('unloaded',None):print(f'模型狀態：{current}，重新要求載入。',flush=True)
        key=os.environ.get('LLAMA24_API_KEY','')
        headers={'Content-Type':'application/json',**({'Authorization':'Bearer '+key} if key else {})}
        request=urllib.request.Request(f'http://127.0.0.1:{_PORT}/models/load',data=json.dumps({'model':alias}).encode(),headers=headers)
        opener=urllib.request.build_opener(urllib.request.ProxyHandler({}))
        with opener.open(request,timeout=10) as response:response.read()
    return False

def _start_model(info,paths):
    exe=Path(info['server']);env=_env(info['library_paths'])
    help_text=_capture([exe,'--help'],env)
    cmd=_command(exe,paths,help_text)
    missing=[arg for arg in cmd[1:] if arg.startswith('--') and arg not in help_text]
    if missing:raise RuntimeError(f'執行檔不支援參數：{missing}')
    bound=False
    for _ in range(25):
        with socket.socket(socket.AF_INET,socket.SOCK_STREAM) as sock:
            try:
                sock.setsockopt(socket.SOL_SOCKET,socket.SO_REUSEADDR,1)
                sock.bind(('127.0.0.1',_PORT))
                bound=True;break
            except OSError:
                time.sleep(0.2)
    if not bound:
        raise RuntimeError(f'連接埠 {_PORT} 無法使用（仍被佔用），請稍候再試。')
    log=_ROOT/'logs'/f'llama-server-{time.time_ns()}.log';(_ROOT/'last-log.txt').write_text(str(log))
    print('▶ 載入模型與 warmup；不是下載或雜湊驗證。',flush=True);print('執行：',shlex.join(cmd),flush=True)
    with log.open('wb') as output:
        process=subprocess.Popen(cmd,cwd=str(exe.parent),env=env,stdin=subprocess.DEVNULL,stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
    saved=_identity(process.pid)
    if saved:(_ROOT/'server-process.json').write_text(json.dumps(saved))
    try:
        deadline=time.monotonic()+_TIMEOUT;report=0
        while time.monotonic()<deadline:
            if process.poll() is not None:
                print(_tail(log));raise RuntimeError(f'llama-server 退出，代碼 {process.returncode}；日誌：{log}')
            try:
                if _health().get('status')=='ok' and process.poll() is None and ('--models-preset' not in cmd or _router_ready()):
                    models=_health('/v1/models');print('✓ 模型服務就緒：',models,flush=True)
                    print(f"API：http://127.0.0.1:{_PORT}/v1\n模型 alias：{_SELECTED['alias']}",flush=True);return process
            except (urllib.error.URLError,OSError,TimeoutError,ValueError):pass
            if time.monotonic()-report>=15:
                print('模型載入／warmup 中…\n'+_tail(log,2000),flush=True);report=time.monotonic()
            time.sleep(2)
        print(_tail(log));raise TimeoutError(f'等待模型逾時 {_TIMEOUT} 秒；日誌：{log}')
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.kill();process.wait()
        (_ROOT/'server-process.json').unlink(missing_ok=True);raise

def _follow_suma(log,process,event,state):
    decoder=codecs.getincrementaldecoder('utf-8')(errors='replace')
    try:
        with Path(log).open('rb') as source:
            while not event.is_set():
                block=source.read(8192)
                if block:
                    state['bytes']+=len(block);text=decoder.decode(block)
                    if text:print(text,end='',flush=True)
                    continue
                if process.poll() is not None:break
                event.wait(0.2)
        text=decoder.decode(b'',final=True)
        if text:print(text,end='',flush=True)
    except Exception as exc:print(f'\n[SUMA] 同步日誌失敗：{exc}；日誌：{log}',flush=True)

def _stop_watcher():
    event=globals().get('_suma_watch_stop')
    if event is not None:event.set()
    thread=globals().get('_suma_watch_thread')
    if thread is not None and thread.is_alive():thread.join(timeout=2)

def _start_suma():
    global suma_process,_suma_watch_stop,_suma_watch_thread
    if _health().get('status')!='ok':raise RuntimeError('模型未就緒，無法啟動 SUMA。')
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:raise RuntimeError('不支援此 SUMA CPU 架構。')
    work=_ROOT/'suma';work.mkdir(parents=True,exist_ok=True)
    tag='0360';name=f'SUMA-Remote-AI-Bridge_0.3.6.0_Beta-linux-{arch}.tar.gz'
    url=f'https://github.com/marchfun1/SUMA-Remote-AI-Bridge/releases/download/{tag}/{name}'
    size=8190363 if arch=='amd64' else None
    archive=_download(url,_ROOT/'downloads'/name,f'SUMA AI Bridge ({tag})',size)
    package=work/f'package-{tag}-{arch}';_extract(archive,package)
    binaries=[path for path in package.rglob('suma-ai-bridge') if path.is_file()]
    if len(binaries)!=1:raise RuntimeError('找不到唯一的 SUMA 主程式。')
    binary=binaries[0].resolve();binary.chmod(binary.stat().st_mode|0o111)
    _stop_watcher();_stop_owned('suma');log=work/'suma-ai-bridge.log'
    if log.exists() and log.stat().st_size:log.rename(work/f'suma-ai-bridge-{time.time_ns()}.log')
    print('\n[SUMA] 自動啟動；以下實際輸出可能含配對碼或 Token，請勿公開。',flush=True)
    with log.open('wb') as output:
        suma_process=subprocess.Popen([str(binary), '--port', str(_PORT), '--auto-approve'], cwd=str(work), stdin=subprocess.DEVNULL, stdout=output, stderr=subprocess.STDOUT, start_new_session=True)
    saved=_identity(suma_process.pid)
    if saved:(_ROOT/'suma-process.json').write_text(json.dumps(saved))
    _suma_watch_stop=threading.Event();state={'bytes':0}
    _suma_watch_thread=threading.Thread(target=_follow_suma,args=(log,suma_process,_suma_watch_stop,state),daemon=True);_suma_watch_thread.start()
    deadline=time.monotonic()+5
    while time.monotonic()<deadline and suma_process.poll() is None:time.sleep(0.2)
    if suma_process.poll() is not None:
        _suma_watch_thread.join(timeout=2);raise RuntimeError(f'SUMA 退出，代碼 {suma_process.returncode}；日誌：{log}')
    print(f'\n[SUMA] 程序仍在執行，PID：{suma_process.pid}。',flush=True)
    if not state['bytes']:print('[SUMA] 目前尚未收到程式輸出。',flush=True)
    print('[SUMA] 程序存活不代表配對／遠端連線成功；請以實際輸出為準。',flush=True)
    return suma_process

def list_downloaded_models():
    print('=== 本機已下載／可供隨時切換的模型 ===', flush=True)
    found = 0
    for name, item in _MODELS.items():
        m_path = _local_model_path(item)
        if m_path.is_file():
            found += 1
            curr = ' [目前運行中]' if name == MODEL else ''
            print(f'✓ {name} ({_human(m_path.stat().st_size)}){curr}', flush=True)
        else:
            print(f'  {name} (尚未下載)', flush=True)
    if not found:
        print('(尚無已下載的模型)', flush=True)
    print('=====================================', flush=True)

def show_suma_log():
    print(_tail(_ROOT/'suma'/'suma-ai-bridge.log',20000),flush=True)

def _is_suma_alive():
    path=_ROOT/'suma-process.json'
    if not path.exists():return False
    try:saved=json.loads(path.read_text())
    except Exception:return False
    pid=saved.get('pid')
    return bool(pid and _identity(pid)==saved)

def stop_model_service():
    _stop_owned('server')
    print('已停止舊模型服務（llama-server），SUMA AI Bridge 保持運行。',flush=True)

def stop_services():
    _stop_watcher();_stop_owned('suma');_stop_owned()
    print('已停止本 Notebook 記錄且身分吻合的程序。',flush=True)

def _run():
    for path in (_ROOT,_ROOT/'downloads',_ROOT/'logs',_ROOT/'metadata'):path.mkdir(parents=True,exist_ok=True)
    print(f'模型：{MODEL}\n載入 MMProj：{LOAD_MMPROJ}\n上下文：{_CTX}\nSUMA：自動啟用\n內容雜湊：不執行',flush=True)
    _stop_watcher();_stop_owned('suma');_stop_owned()
    info=_ensure_llama();paths=_prepare_files()
    process=_start_model(info,paths)
    try:_start_suma()
    except Exception:
        print('[SUMA] 啟動失敗；模型服務仍在執行，錯誤如下。',flush=True);raise
    return process

server_process = _run()


## 後續下載與切換模型

此表單提供三個選單：新模型、載入 MMProj、上下文（64K／128K／256K）。選好後按執行即可。

- 不會刪除先前已下載的模型；新下載的模型會一併整理至同一個模型目錄（`active_models`），讓 `llama.cpp` 可隨時於多個已下載模型間切換。
- 下載／準備完成後僅停止舊模型並載入新模型；SUMA AI Bridge 持續保持運行，無須重新配對。
- 載入新模型失敗時舊服務已停止，模型檔案仍保留；可在此表單改選舊模型或調小上下文重試。


In [ ]:
# @title 下載其他模型並切換 llama.cpp
NEW_MODEL = "Qwen3.8-27B UD-Q2_K_XL" # @param ["Gemma 4 12B QAT Q4_0", "Qwen3.5-9B Q5_K_M", "Qwen3.8-27B UD-Q2_K_XL", "Inkling-2-Mini UD-Q2_K_XL"]
MMPROJ = True # @param {type:"boolean"}
CONTEXT_SIZE = "64K" # @param ["64K", "128K", "256K"]

def download_and_switch(model, mmproj, context):
    global MODEL, LOAD_MMPROJ, CONTEXT, _CTX, _SELECTED, server_process
    required = ("_MODELS", "_ROOT", "_ensure_llama", "_prepare_files",
                "_start_model", "_start_suma", "stop_services", "stop_model_service", "_sync_models_dir", "list_downloaded_models", "MODEL",
                "LOAD_MMPROJ", "CONTEXT", "_CTX", "_SELECTED")
    if any(name not in globals() for name in required):
        raise RuntimeError('請先在同一執行階段執行上方的主要程式格。')
    allowed = ("Gemma 4 12B QAT Q4_0", "Qwen3.5-9B Q5_K_M",
               "Qwen3.8-27B UD-Q2_K_XL", "Inkling-2-Mini UD-Q2_K_XL")
    contexts = {"64K":65536, "128K":131072, "256K":262144}
    if model not in allowed or model not in _MODELS or context not in contexts:
        raise ValueError('請使用表單提供的模型與上下文選項。')
    if not isinstance(mmproj, bool):
        raise ValueError("MMProj parameter must be boolean.")
    previous = (MODEL, LOAD_MMPROJ, CONTEXT, _CTX, _SELECTED)
    MODEL, LOAD_MMPROJ, CONTEXT = model, mmproj, context
    _CTX, _SELECTED = contexts[context], _MODELS[model]
    print(f'準備模型：{MODEL}｜MMProj：{mmproj}｜上下文：{_CTX}', flush=True)
    try:
        for path in (_ROOT, _ROOT/"downloads", _ROOT/"logs", _ROOT/"metadata"):
            path.mkdir(parents=True, exist_ok=True)
        print('① 準備新模型；暫不停止原服務。', flush=True)
        info = _ensure_llama()
        paths = _prepare_files()
    except BaseException:
        MODEL, LOAD_MMPROJ, CONTEXT, _CTX, _SELECTED = previous
        print('準備失敗或已取消；原服務未被停止，設定已還原。', flush=True)
        raise
    print('② 檔案已就緒，停止舊服務並載入新模型。', flush=True)
    stop_model_service()
    time.sleep(1)
    server_process = None
    try:
        server_process = _start_model(info, paths)
    except BaseException:
        print('載入失敗；舊服務已停止，模型檔案仍保留。請改選原模型或降低上下文後重試。', flush=True)
        raise
    if _is_suma_alive():
        print('③ 新模型 API 已就緒；SUMA AI Bridge 仍在持續運行，保持原有連線，無需重啟。', flush=True)
    else:
        print('③ 新模型 API 已就緒；偵測到 SUMA 未運行，自動啟動 SUMA。', flush=True)
        try:
            _start_suma()
        except BaseException:
            print('SUMA 啟動未完成；新模型服務仍在運行，請查看日誌。', flush=True)
            raise
    list_downloaded_models()
    print(f"已切換：{MODEL}\nAPI：http://127.0.0.1:{_PORT}/v1\n模型 alias：{_SELECTED['alias']}", flush=True)
    print('SUMA 配對／連線狀態請以其實際輸出為準。', flush=True)
    return server_process

server_process = download_and_switch(NEW_MODEL, MMPROJ, CONTEXT_SIZE)


In [ ]:
# @title 🔍 連線檢查與重啟服務
# @markdown 當 Colab 連線中斷後恢復，可執行此區塊檢查檔案與服務狀態，必要時自動拉起或重啟。

import json
import os
import signal
import socket
import subprocess
import time
import urllib.request
from pathlib import Path

_ROOT = Path('/content/llama-direct-b12651') if Path('/content').exists() else Path.home()/'llama-direct-b12651'
_MODELS_ROOT = Path('/content/models') if Path('/content').exists() else Path.home()/'models'
_PORT = 8081

print("=" * 60)
print('🔍 正在檢查 Colab 執行階段與下載檔案狀態...')
print("=" * 60)

# 1. 檢查根目錄與下載模型是否被清空
models_dir = Path('/content/models') if Path('/content').exists() else Path.home()/'models'
packages_dir = _ROOT / 'packages'
existing_models = list(models_dir.glob('**/*.gguf')) if models_dir.exists() else []

if not _ROOT.exists() or (not existing_models and not packages_dir.exists()):
    print('❌ 【警告】偵測到暫存與模型內容已被 Colab 清空！')
    print('   可能原因：Colab 執行階段曾完全斷線或被 Google 回收釋放 GPU。')
    print('👉 請回到上方【啟動執行表單】重新執行完整初始化與下載。')
else:
    print(f'✅ 檔案系統完整：已找到 {len(existing_models)} 個 GGUF 模型檔案。')
    for m in existing_models:
        size_gb = m.stat().st_size / (1024 ** 3)
        print(f"   - {m.name} ({size_gb:.2f} GB)")

    # 2. 檢查 llama-server 服務狀態
    def check_server_status():
        try:
            req = urllib.request.Request(f'http://127.0.0.1:{_PORT}/health')
            opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
            with opener.open(req, timeout=3) as resp:
                data = json.load(resp)
                return data.get('status') == 'ok'
        except Exception:
            return False

    server_ok = check_server_status()
    if server_ok:
        print(f'✅ llama.cpp 模型服務正常運行中 (http://127.0.0.1:{_PORT}/v1)。')
    else:
        print('⚠️ llama.cpp 服務目前未回應。')

    # 3. 檢查 SUMA AI Bridge
    suma_proc_file = _ROOT / 'suma-process.json'
    suma_alive = False
    if suma_proc_file.exists():
        try:
            saved = json.loads(suma_proc_file.read_text())
            pid = saved.get('pid')
            if pid:
                os.kill(pid, 0)
                suma_alive = True
        except (OSError, Exception):
            suma_alive = False

    if suma_alive:
        print('✅ SUMA AI Bridge 程序正常運行中。')
    else:
        print('⚠️ SUMA AI Bridge 尚未運行或已中斷。')

    # 4. 成功確認或拉起後，自動進入常駐保活模式，避免再次閒置中斷
    if server_ok and suma_alive:
        print('\n🎉 所有服務均在正常運行中，連線完好無損！')
        print('🛡️ 現自動進入常駐保活狀態，防止 Colab 再次閒置中斷...')
        import datetime, time
        cycle = 0
        try:
            while True:
                time.sleep(300)
                cycle += 5
                now_str = datetime.datetime.now().strftime("%H:%M:%S")
                print(f'[{now_str}] 💓 保活心跳回報：已持續穩定運行 {cycle} 分鐘 | 連線與 GPU 保持活躍中', flush=True)
        except KeyboardInterrupt:
            print('\n🛑 已手動停止常駐保活監控。', flush=True)
    else:
        print('\n🔄 正在嘗試重新拉起中斷的服務...')
        try:
            if 'info' in globals() and 'paths' in globals() and '_start_model' in globals():
                if not server_ok:
                    print('  - 重新啟動模型服務中...')
                    server_process = _start_model(info, paths)
                if not suma_alive and '_start_suma' in globals():
                    print('  - 重新啟動 SUMA AI Bridge 中...')
                    _start_suma()
                print('✅ 服務已成功重新拉起！')
                print('🛡️ 現自動進入常駐保活狀態，防止 Colab 再次閒置中斷...')
                import datetime, time
                cycle = 0
                try:
                    while True:
                        time.sleep(300)
                        cycle += 5
                        now_str = datetime.datetime.now().strftime("%H:%M:%S")
                        print(f'[{now_str}] 💓 保活心跳回報：已持續穩定運行 {cycle} 分鐘 | 連線與 GPU 保持活躍中', flush=True)
                except KeyboardInterrupt:
                    print('\n🛑 已手動停止常駐保活監控。', flush=True)
            else:
                print('ℹ️ 核心變數已重置，但下載檔案依然保留在磁碟中。')
                print('👉 請直接重新執行上方【啟動執行表單】，程式會自動使用已下載的模型與二進位檔，無須重新下載！')
        except Exception as e:
            print(f'❌ 自動拉起失敗：{e}')
            print('👉 請直接重新點選上方【啟動執行表單】執行，程式會自動跳過已下載完成之檔案。')
